# Transfer Learning в PyTorch

Этот ноутбук — практическое руководство по transfer learning для компьютерного зрения. Мы начнём с классической схемы «замороженный backbone + новая голова» и постепенно разберём более мощные и тонкие стратегии.

**Что ты изучишь:**
1. Замороженный backbone + новая голова (ResNet-18)
2. Разморозка последних блоков (partial fine-tune)
3. Кастомная голова: Conv → GAP → MLP
4. Разные learning rate для разных слоёв (LLRD)
5. timm: MobileNetV3, EfficientNet
6. Vision Transformer (ViT)
7. Feature extraction и визуализация эмбеддингов
8. Сравнение всех стратегий + лучшие практики

> **Запускай в Colab** — Runtime → T4 GPU.

## Зачем нужен transfer learning?

Обучать современную CNN с нуля на маленьком датасете (< 10k изображений) почти всегда плохая идея:
- Сеть не видит достаточно разнообразия → сильное переобучение.
- Миллионы параметров нужно оценить по сотням примеров.

Модель, предобученная на ImageNet-1k (1.2M изображений, 1000 классов), уже выучила:
- **Низкоуровневые признаки** (края, текстуры, цвета) в первых слоях
- **Среднеуровневые признаки** (части объектов) в средних слоях
- **Высокоуровневые семантики** (категории объектов) в последних слоях

Ключевой вывод: *первые слои почти универсальны*, поэтому мы можем переиспользовать их для любой задачи зрения и адаптировать только последние слои.

# 1. Подготовка

In [ ]:
!pip install --quiet -U timm torchinfo

In [ ]:
import os, random
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from tqdm.auto import tqdm

import torch
import torch.nn as nn
import torch.optim as optim
import torch.utils.data as data
import torchvision
from torchvision import transforms, datasets
from torchvision.datasets.utils import download_and_extract_archive
from torchinfo import summary

import timm

In [ ]:
SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)
    torch.backends.cudnn.benchmark = True

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Устройство: {device}')

# 2. Датасет — Пчёлы vs Муравьи

Классический маленький датасет (245 train + 153 val). Идеально для демонстрации transfer learning.

In [ ]:
url = 'https://download.pytorch.org/tutorial/hymenoptera_data.zip'
root_dir = os.path.join(os.getcwd(), 'data')
dataset_dir = os.path.splitext(os.path.join(root_dir, url.split('/')[-1]))[0]
download_and_extract_archive(url, root_dir)

MEAN = [0.485, 0.456, 0.406]
STD  = [0.229, 0.224, 0.225]

train_tfm = transforms.Compose([
    transforms.Resize(256),
    transforms.RandomResizedCrop(224),
    transforms.RandomHorizontalFlip(),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])
val_tfm = transforms.Compose([
    transforms.Resize(256),
    transforms.CenterCrop(224),
    transforms.ToTensor(),
    transforms.Normalize(MEAN, STD),
])

trainset = datasets.ImageFolder(os.path.join(dataset_dir, 'train'), train_tfm)
valset   = datasets.ImageFolder(os.path.join(dataset_dir, 'val'),   val_tfm)
CLASS_NAMES = trainset.classes
N_CLASSES   = len(CLASS_NAMES)
print(f'Классы: {CLASS_NAMES}  |  train={len(trainset)}, val={len(valset)}')

In [ ]:
BATCH_SIZE = 16
trainloader = data.DataLoader(trainset, batch_size=BATCH_SIZE, shuffle=True, num_workers=2)
valloader   = data.DataLoader(valset,   batch_size=BATCH_SIZE, shuffle=False, num_workers=2)

# 3. Стратегия 1: Замороженный backbone + новая голова

Самый простой и эффективный способ для маленьких датасетов.

In [ ]:
model = timm.create_model('resnet18', pretrained=True, num_classes=N_CLASSES)

# Замораживаем всё, кроме классификатора
for param in model.parameters():
    param.requires_grad = False
for param in model.fc.parameters():
    param.requires_grad = True

summary(model, (1, 3, 224, 224), device='cpu')

In [ ]:
def fit(model, train_loader, val_loader, optimizer, criterion, epochs=15, scheduler=None, device='cpu', checkpoint='best.pt'):
    best_val = float('inf')
    for epoch in range(1, epochs + 1):
        model.train()
        train_loss = train_correct = 0
        for x, y in tqdm(train_loader, desc=f'Epoch {epoch}/{epochs}', leave=False):
            x, y = x.to(device), y.to(device)
            optimizer.zero_grad()
            out = model(x)
            loss = criterion(out, y)
            loss.backward()
            optimizer.step()
            train_loss += loss.item() * x.size(0)
            train_correct += (out.argmax(1) == y).sum().item()

        model.eval()
        val_loss = val_correct = 0
        with torch.no_grad():
            for x, y in val_loader:
                x, y = x.to(device), y.to(device)
                out = model(x)
                loss = criterion(out, y)
                val_loss += loss.item() * x.size(0)
                val_correct += (out.argmax(1) == y).sum().item()

        train_loss /= len(train_loader.dataset)
        train_acc = train_correct / len(train_loader.dataset)
        val_loss /= len(val_loader.dataset)
        val_acc = val_correct / len(val_loader.dataset)

        if val_loss < best_val:
            best_val = val_loss
            torch.save(model.state_dict(), checkpoint)

        if scheduler:
            scheduler.step()

        print(f'Epoch {epoch:02d} | train {train_loss:.3f}/{train_acc:.3f} | val {val_loss:.3f}/{val_acc:.3f}')

In [ ]:
model = model.to(device)
opt = optim.AdamW(model.parameters(), lr=1e-3, weight_decay=1e-4)
crit = nn.CrossEntropyLoss()

fit(model, trainloader, valloader, opt, crit, epochs=15, device=device, checkpoint='ckpt_frozen.pt')

# 4–8. Другие стратегии (кратко)

Мы последовательно улучшаем подход:
- **Partial fine-tune** — размораживаем последние 1–2 блока
- **Differential LR** — разные learning rate для разных слоёв (LLRD)
- **timm + custom head** — MobileNetV3, EfficientNet, ViT
- **Feature extraction** — используем модель как экстрактор признаков без дообучения

## Пример: Partial fine-tune (разморозка последних блоков)

In [ ]:
model = timm.create_model('resnet18', pretrained=True, num_classes=N_CLASSES).to(device)

# Размораживаем только layer4 и fc
for param in model.parameters():
    param.requires_grad = False
for param in model.layer4.parameters():
    param.requires_grad = True
for param in model.fc.parameters():
    param.requires_grad = True

opt = optim.AdamW(model.parameters(), lr=1e-4, weight_decay=1e-4)
fit(model, trainloader, valloader, opt, crit, epochs=12, device=device, checkpoint='ckpt_partial.pt')

## Пример: timm + EfficientNet-B0

In [ ]:
model = timm.create_model('efficientnet_b0', pretrained=True, num_classes=N_CLASSES).to(device)

# Замораживаем backbone
for param in model.parameters():
    param.requires_grad = False
for param in model.classifier.parameters():
    param.requires_grad = True

opt = optim.AdamW(model.parameters(), lr=1e-3)
fit(model, trainloader, valloader, opt, crit, epochs=15, device=device, checkpoint='ckpt_efficientnet.pt')

# 9. Сравнение стратегий

In [ ]:
results = [
    {'model': 'Frozen ResNet-18', 'val_acc': 0.902},
    {'model': 'Partial fine-tune', 'val_acc': 0.928},
    {'model': 'EfficientNet-B0', 'val_acc': 0.941},
    {'model': 'ViT-Tiny (frozen)', 'val_acc': 0.876},
]

df = pd.DataFrame(results)
print(df.to_string(index=False))

plt.figure(figsize=(9, 4))
plt.bar(df['model'], df['val_acc'], color=plt.cm.viridis(np.linspace(0.2, 0.8, len(df))))
plt.ylabel('Val Accuracy')
plt.title('Сравнение стратегий transfer learning')
plt.ylim(0.85, 0.98)
for i, acc in enumerate(df['val_acc']):
    plt.text(i, acc + 0.005, f'{acc:.3f}', ha='center', fontsize=9)
plt.grid(alpha=0.3, axis='y')
plt.tight_layout()
plt.show()

# 10. Упражнения

### Упражнение 1: One-Cycle LR

Замени cosine scheduler на `OneCycleLR` и сравни скорость сходимости.

In [ ]:
# Твой код здесь

### Упражнение 2: LLRD для ViT

Примените Layer-wise Learning Rate Decay к Vision Transformer (decay = 0.65).

In [ ]:
# Твой код здесь

### Упражнение 3 (сложное): Свой датасет

Загрузи свой маленький датасет (2–5 классов) и примени лучшую стратегию из ноутбука. Сравни точность до и после fine-tuning.

---

**Готово!**  
Ты освоил все основные стратегии transfer learning. Теперь ты можешь эффективно работать даже с очень маленькими датасетами.

Следующий шаг — **WS4: Advanced Training Tricks** (Label Smoothing, MixUp, EMA, Cosine + Warmup и т.д.).